# Descriptive Statistics and Welch's T-Test for Curriculum Year

In [ ]:
import pandas as pd
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

## 1. Load the Dataset

In [ ]:
try:
    df = pd.read_csv('../data/social_work_exam_dataset_curri.csv')
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Error: The file '../data/social_work_exam_dataset_curri.csv' was not found.")

## 2. Initial Data Exploration

In [ ]:
print("First 5 rows of the dataset:")
display(df.head())

print("\nDataset Information:")
df.info()

## 3. Handle Missing or Non-Numeric Values
Before performing statistical analysis, we need to clean the data. We'll convert columns to numeric where possible and handle any resulting missing values.

In [ ]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.fillna(df.median(), inplace=True)

print("Data cleaned and missing values handled.")

## 4. Descriptive Statistics

In [ ]:
print("Descriptive Statistics for the entire dataset:")
display(df.describe().T)

## 5. Analysis by Curriculum Year

In [ ]:
print("Summary Statistics for Exam Result Percent by Curriculum Year:")
summary_stats = df.groupby('Curriculum_Year')['ExamResultPercent'].agg(['count', 'mean', 'median', 'std'])
display(summary_stats)

print("\nDetailed Descriptive Statistics by Curriculum Year:")
curriculum_groups = df.groupby('Curriculum_Year')
for year, group in curriculum_groups:
    print(f"--- Curriculum Year: {int(year)} ---")
    display(group.describe().T)

## 6. Welch's T-Test
We will use Welch's t-test to compare the means of two independent groups (Curriculum Year 2010 vs. 2017) without assuming equal variances. This helps determine if there are statistically significant differences in key academic and performance metrics between the two curricula.

In [ ]:
group1 = df[df['Curriculum_Year'] == 2010]
group2 = df[df['Curriculum_Year'] == 2017]

features_to_compare = [
    'GPA', 'MockExamScore', 'StudyHours', 'ExamResultPercent', 
    'InternshipGrade', 'Confidence', 'TestAnxiety', 'MotivationScore'
]

ttest_results = []

print("--- Welch's T-Test Results (2010 vs 2017) ---\n")
for feature in features_to_compare:
    t_stat, p_val = ttest_ind(group1[feature].dropna(), group2[feature].dropna(), equal_var=False)
    
    result = {
        'Feature': feature,
        'T-statistic': t_stat,
        'P-value': p_val,
        'Significant': 'Yes' if p_val < 0.05 else 'No'
    }
    ttest_results.append(result)

results_df = pd.DataFrame(ttest_results)
display(results_df)

## 7. Visualizations
Box plots are used to visualize the distribution of key features between the two curriculum years, providing a clear visual comparison.

In [ ]:
for feature in features_to_compare:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='Curriculum_Year', y=feature, data=df, palette='viridis')
    plt.title(f'{feature} by Curriculum Year', fontsize=16, fontweight='bold')
    plt.xlabel('Curriculum Year', fontsize=12)
    plt.ylabel(feature, fontsize=12)
    plt.show()

### Correlation Heatmap
A heatmap is used to visualize the correlation matrix of the features, helping to identify relationships between them.

In [ ]:
plt.figure(figsize=(18, 15))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix of Features', fontsize=20, fontweight='bold')
plt.show()